# _comum — leitura, padrão visual, metadados e qualidade
Este notebook não é para ser lido sozinho: os outros o executam com `%run ./_comum`. Ele garante que todos usem a mesma
leitura dos dados, o mesmo padrão de gráficos, os mesmos metadados (seção 3 do 01) e a mesma verificação de qualidade
(seção 4.1 do 01).

In [ ]:
# 0. Limita as threads de cálculo a 1. No Serverless gratuito, a máquina "mostra" muitos núcleos mas entrega poucos;
#    o numpy e o scikit-learn abrem uma thread por núcleo, elas disputam a CPU e as contas ficam muito lentas ou travam.
import os
try:
    from threadpoolctl import threadpool_limits
    threadpool_limits(limits=1)
    print(f"Threads de cálculo limitadas a 1 (a máquina informa {os.cpu_count()} núcleos).")
except ImportError:
    print("threadpoolctl não encontrado: as threads de cálculo não foram limitadas.")

# 1. Leitura dos CSVs como texto (preserva 'unknown' e 'null' para o dicionário)
import os
import pandas as pd
import numpy as np

# encontra os CSVs do hackathon subindo a partir da pasta do notebook (funciona para qualquer pessoa do grupo):
# na raiz do repositório (como no repositório do hackathon) ou numa pasta "dados" dentro dela
caminho_nb = "/Workspace" + dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
raiz = os.path.dirname(caminho_nb)
while raiz != "/Workspace":
    if os.path.isfile(os.path.join(raiz, "history.csv")):
        PASTA = raiz
        break
    if os.path.isfile(os.path.join(raiz, "dados", "history.csv")):
        PASTA = os.path.join(raiz, "dados")
        break
    raiz = os.path.dirname(raiz)
assert raiz != "/Workspace", "history.csv não encontrado: deixe os CSVs na raiz do repositório ou numa pasta dados/"
print("Lendo os dados de:", PASTA)

hist_raw = pd.read_csv(f"{PASTA}/history.csv", dtype=str, keep_default_na=False)
inf_raw = pd.read_csv(f"{PASTA}/inference.csv", dtype=str, keep_default_na=False)

# conferência: registros reais x linhas físicas do arquivo (a diferença são as quebras de linha no feedback)
for nome, df in [("history.csv", hist_raw), ("inference.csv", inf_raw)]:
    with open(f"{PASTA}/{nome}", encoding="utf-8") as f:
        linhas_fisicas = sum(1 for _ in f) - 1          # menos o cabeçalho
    print(f"{nome}: {len(df):,} clientes · {df.shape[1]} colunas · {linhas_fisicas:,} linhas físicas no arquivo"
          .replace(",", "."))

assert hist_raw.shape == (5634, 23) and inf_raw.shape == (1409, 22), "leitura diferente do esperado"
assert set(inf_raw.columns) == set(hist_raw.columns) - {"Churn"}, "colunas da inferência diferentes do histórico"

In [ ]:
# 1.1 Padrão visual dos gráficos (rodar antes de qualquer gráfico)
#     Princípios: título que afirma o achado,
#     uma cor de base e uma de destaque, rótulos diretos, eixos limpos.
#     Convenção: nomes originais das colunas nos gráficos e tradução no rodapé.
import matplotlib.pyplot as plt
import textwrap

COR, DESTAQUE, CINZA, TEXTO = "#1f6f8b", "#d9822b", "#B0B0B0", "#333333"

plt.rcParams.update({
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#999999", "axes.titlelocation": "left", "axes.titleweight": "bold",
    "axes.titlesize": 11, "figure.dpi": 110,
})

TRADUCAO = {
    "customerID": "identificador do cliente", "gender": "sexo", "SeniorCitizen": "idoso, 65 anos ou mais (1 = sim)",
    "Partner": "tem parceiro(a)", "Dependents": "tem dependentes", "tenure": "tempo de casa, em meses",
    "PhoneService": "telefonia", "MultipleLines": "múltiplas linhas", "InternetService": "tipo de internet",
    "OnlineSecurity": "segurança online", "OnlineBackup": "backup online",
    "DeviceProtection": "proteção de aparelho", "TechSupport": "suporte técnico",
    "StreamingTV": "streaming de TV", "StreamingMovies": "streaming de filmes",
    "Contract": "tipo de contrato", "PaperlessBilling": "fatura digital", "PaymentMethod": "forma de pagamento",
    "MonthlyCharges": "mensalidade, em R$", "TotalCharges": "total pago, em R$",
    "MonthlyIncome": "renda familiar mensal, em R$", "Churn": "cancelou", "CustomerFeedback": "comentário do cliente",
}

def virg(x, d=2):
    """Número com vírgula decimal e ponto de milhar."""
    return f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")

def rodape(fig, colunas, fonte="history.csv"):
    """Escreve no pé da figura a tradução das colunas exibidas e a fonte dos dados."""
    itens = " · ".join(f"{c} = {TRADUCAO[c]}" for c in colunas if c in TRADUCAO)
    largura = int(fig.get_figwidth() * 15)                    # quebra o texto conforme a largura da figura
    texto = textwrap.fill(f"Variáveis: {itens}.", width=largura) + f"\nFonte: {fonte}."
    texto = texto.replace("$", r"\$")                          # evita que o "$" de R$ vire fórmula matemática
    fig.text(0.01, 0.005, texto, ha="left", va="bottom", fontsize=8, color="#666666")
    n_linhas = texto.count("\n") + 1
    reserva = (n_linhas * 0.16 + 0.12) / fig.get_figheight() # espaço do rodapé, em fração da altura
    plt.tight_layout(rect=(0, reserva, 1, 1))

print("Padrão visual carregado.")

In [ ]:
CATALOG, SCHEMA = "workspace", "terra_signal"
FQ = f"{CATALOG}.{SCHEMA}"

In [ ]:
# 3. Metadados: conhecimento de negócio e classificação estatística de cada coluna (antes de olhar os dados)
SIM_NAO = ["Yes", "No"]
COM_INTERNET = ["Yes", "No", "No internet service"]

meta = pd.DataFrame([
    # coluna,            grupo,          tipo_estatistico,          papel,     descricao,                                          unidade,  dominio_esperado
    ("customerID",       "Identificador", "identificador",          "id",      "Identificador único do cliente (0000-XXXXX)",       "",       "texto único"),
    ("gender",           "Cliente",       "qualitativa binária",    "feature", "Sexo do cliente",                                   "",       ["Male", "Female"]),
    ("SeniorCitizen",    "Cliente",       "qualitativa binária",    "feature", "Cliente idoso (65 anos ou mais): 1 = sim, 0 = não", "",       ["0", "1"]),
    ("Partner",          "Cliente",       "qualitativa binária",    "feature", "Possui cônjuge ou parceiro(a)",                     "",       SIM_NAO),
    ("Dependents",       "Cliente",       "qualitativa binária",    "feature", "Possui dependentes",                                "",       SIM_NAO),
    ("MonthlyIncome",    "Cliente",       "quantitativa contínua",  "feature", "Renda familiar mensal (base externa)",              "R$",     (0, None)),
    ("tenure",           "Contrato",      "quantitativa discreta",  "feature", "Tempo de casa (teto real de 72 meses)",             "meses",  (0, 72)),
    ("Contract",         "Contrato",      "qualitativa ordinal",    "feature", "Tipo de contrato (fidelidade crescente)",           "",       ["Month-to-month", "One year", "Two year"]),
    ("PaperlessBilling", "Contrato",      "qualitativa binária",    "feature", "Fatura digital (sem papel)",                        "",       SIM_NAO),
    ("PaymentMethod",    "Contrato",      "qualitativa nominal",    "feature", "Forma de pagamento",                                "",       ["Electronic check", "Mailed check",
                                                                                                                                                 "Bank transfer (automatic)", "Credit card (automatic)"]),
    ("PhoneService",     "Serviços",      "qualitativa binária",    "feature", "Possui telefonia",                                  "",       SIM_NAO),
    ("MultipleLines",    "Serviços",      "qualitativa nominal",    "feature", "Múltiplas linhas (No phone service = sem telefonia)", "",     ["Yes", "No", "No phone service"]),
    ("InternetService",  "Serviços",      "qualitativa nominal",    "feature", "Tipo de internet",                                  "",       ["DSL", "Fiber optic", "No"]),
    ("OnlineSecurity",   "Serviços",      "qualitativa nominal",    "feature", "Segurança online",                                  "",       COM_INTERNET),
    ("OnlineBackup",     "Serviços",      "qualitativa nominal",    "feature", "Backup online",                                     "",       COM_INTERNET),
    ("DeviceProtection", "Serviços",      "qualitativa nominal",    "feature", "Proteção de aparelho",                              "",       COM_INTERNET),
    ("TechSupport",      "Serviços",      "qualitativa nominal",    "feature", "Suporte técnico",                                   "",       COM_INTERNET),
    ("StreamingTV",      "Serviços",      "qualitativa nominal",    "feature", "Streaming de TV",                                   "",       COM_INTERNET),
    ("StreamingMovies",  "Serviços",      "qualitativa nominal",    "feature", "Streaming de filmes",                               "",       COM_INTERNET),
    ("MonthlyCharges",   "Financeiro",    "quantitativa contínua",  "feature", "Valor do último mês faturado, com taxas",          "R$",     (0, None)),
    ("TotalCharges",     "Financeiro",    "quantitativa contínua",  "feature", "Total acumulado pago (soma das mensalidades)",     "R$",     (0, None)),
    ("CustomerFeedback", "Texto",         "texto livre",            "texto",   "Comentário do cliente sobre o serviço (inglês)",    "",       "texto livre"),
    ("Churn",            "Alvo",          "qualitativa binária",    "alvo",    "Cliente cancelou",                                  "",       SIM_NAO),
], columns=["coluna", "grupo", "tipo_estatistico", "papel", "descricao", "unidade", "dominio_esperado"]).set_index("coluna")

# de onde vem a certeza sobre cada descrição: verificado nos dados, confirmado pelos organizadores, evidente pelo nome
# e pelos valores, ou premissa. Respostas dos organizadores: Carlos (Databricks), em 10/11
CONFIRMACAO = {
    "customerID": "Verificado: valores únicos e sem repetição entre as bases (4.2)",
    "tenure": "Organizadores: meses, com teto real de 72 (a empresa tem cerca de 7 anos); o texto cita '70 months'",
    "MonthlyCharges": "Organizadores: valor do último mês faturado, em R$, com taxas e adicionais; para quem saiu, "
                      "o último mês antes do cancelamento",
    "TotalCharges": "Organizadores: total pago, soma das mensalidades; o preço muda com plano e promoções, por isso "
                    "fica só a ±20% de MonthlyCharges × tenure (4.2)",
    "MultipleLines": "Verificado: 'No phone service' só aparece para quem não tem telefonia (4.2)",
    "Churn": "Verificado: quem tem Yes conta no texto que cancelou (7.3)",
    "CustomerFeedback": "Verificado: texto em inglês, lido nos exemplos (7.3)",
    "SeniorCitizen": "Organizadores: 1 = idoso; sem data de nascimento, considerar 65 anos ou mais",
    "PhoneService": "Organizadores: 'yes'/'no' em minúsculo é só erro de padronização, mesma fonte",
    "MonthlyIncome": "Organizadores: renda familiar mensal em R$, de base externa, sem garantia de precisão; "
                     "pode ser ignorada se não ajudar",
}
for s in ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]:
    CONFIRMACAO[s] = "Verificado: 'No internet service' só aparece para quem não tem internet (4.2)"
meta["confirmacao"] = meta.index.map(CONFIRMACAO).fillna("Evidente pelo nome da coluna e pelos valores")

# conferências: toda coluna da base tem metadado, e nenhum metadado sobra
faltando = set(hist_raw.columns) - set(meta.index)
sobrando = set(meta.index) - set(hist_raw.columns)
print("Colunas sem metadado:", faltando or "nenhuma")
print("Metadados sem coluna:", sobrando or "nenhum")

In [ ]:
# 4.1 Verificação de qualidade: cada valor é comparado com o domínio esperado da seção 3
def verificar_coluna(s: pd.Series, dominio):
    """Devolve (n_vazios, n_fora_do_dominio, exemplos de valores fora) para uma coluna lida como texto."""
    s = s.astype(str)
    vazios = s.str.strip().eq("")
    if isinstance(dominio, list):                         # qualitativa: lista de categorias aceitas
        fora = ~s.isin(dominio) & ~vazios
    elif isinstance(dominio, tuple):                      # quantitativa: intervalo (mínimo, máximo)
        x = pd.to_numeric(s, errors="coerce")
        lo, hi = dominio
        fora = (x.isna() & ~vazios) | (x < lo) | ((x > hi) if hi is not None else False)
    else:                                                 # identificador ou texto livre: só vazios
        fora = pd.Series(False, index=s.index)
    exemplos = ", ".join(f"{v} ({n})" for v, n in s[fora].value_counts().head(4).items())
    return int(vazios.sum()), int(fora.sum()), exemplos

linhas = []
for col, m in meta.iterrows():
    for base, df in [("historico", hist_raw), ("inferencia", inf_raw)]:
        if col not in df.columns:
            continue
        n_vaz, n_fora, ex = verificar_coluna(df[col], m["dominio_esperado"])
        linhas.append({"coluna": col, "base": base, "tipo_estatistico": m["tipo_estatistico"],
                       "n": len(df), "vazios": n_vaz, "fora_do_dominio": n_fora,
                       "pct_problema": round(100 * (n_vaz + n_fora) / len(df), 2), "exemplos": ex})
qualidade = pd.DataFrame(linhas)